In [ ]:
'''
corrected QNI inspired from paper and QNI_ch4 code

model saved: qni_ccp-3
'''

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import random
import os

# ========== SEEDING ==========
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)

# ========== DEVICE ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== CONFIGURATION ==========
n_qubits = 10
batch_size = 16
num_classes = 26
num_epochs = 70
lr = 0.0005

# QNI-CCP Specific Config
EPSILON_ALPHA = 1.0
EPSILON_BETA = 1.0
# Assuming standard circuit depth/gates if not dynamically calculated per layer
CIRCUIT_DEPTH = 6  # Based on weight_shapes (6 layers)
N_CNOTS = 6 * (n_qubits - 1) # Approximate CNOT count

# ========== DATA TRANSFORMS ==========
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ========== DATASET LOADING ==========
TRAIN_PATH = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/train'
VAL_PATH   = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/val'
TEST_PATH  = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH, transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH, transform=eval_transform)
    print("** Dataset loaded successfully **")
    
except Exception as e:
    print(f"Error loading datasets: {e}")
    # Fallback for testing logic
    os.makedirs("dummy_data/train/class1", exist_ok=True)
    os.makedirs("dummy_data/val/class1", exist_ok=True)
    os.makedirs("dummy_data/test/class1", exist_ok=True)
    train_dataset = ImageFolder("dummy_data/train", transform=train_transform)
    val_dataset = ImageFolder("dummy_data/val", transform=eval_transform)
    test_dataset = ImageFolder("dummy_data/test", transform=eval_transform)

# ========== CLASS WEIGHTS ==========
try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(class_weight='balanced',
                                         classes=np.unique(labels),
                                         y=labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print(f"Class weights calculated.")
    
except:
    print("Warning: Could not calculate class weights. Using ones.")
    class_weights_tensor = torch.ones(num_classes).to(device)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# ========== QUANTUM CIRCUIT ==========
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Angle Embedding
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    
    # Basic Entangler Layers
    for l in range(weights.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (6, n_qubits)}

# ========== MODEL ARCHITECTURE ==========
class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),    
            nn.BatchNorm2d(8), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),   
            nn.BatchNorm2d(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  
            nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), 
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 224, 3, stride=2, padding=1), 
            nn.BatchNorm2d(224), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                
        )
        self.fc_expand = nn.Linear(224, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc_expand(x))
        return self.fc_project(x)

class HybridQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)
        
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.tanh(x)
        q_out = self.q_layer(x)
        return self.classifier(q_out)

# ========== DYNAMIC FOCAL LOSS ==========
class ScheduledFocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=0, reduction='mean'):
        super(ScheduledFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=class_weights_tensor)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ========== QNI-CCP HELPER FUNCTIONS ==========

def compute_quantum_epsilon(n_cnots=54, depth=6, alpha=1.0, beta=1.0):
    """
    Compute epsilon_q based on quantum circuit complexity.
    """
    epsilon_q = 1.0 / (1 + alpha * n_cnots + beta * depth)
    return epsilon_q

def compute_class_centroids(model, loader, device, num_classes):
    """
    Compute class centroids in the FEATURE SPACE (before quantum layer)
    """
    model.eval()
    sums = torch.zeros(num_classes, n_qubits, device=device)
    counts = torch.zeros(num_classes, device=device)
    
    print("Computing class centroids...")
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            # Get features before quantum layer (after feature extractor)
            features = model.feature_extractor(x)
            features = torch.tanh(features)  # Apply tanh as in forward pass
            
            for c in range(num_classes):
                mask = (y == c)
                if mask.any():
                    sums[c] += features[mask].sum(0)
                    counts[c] += mask.sum()
    
    # Avoid division by zero
    counts[counts == 0] = 1
    centroids = sums / counts.unsqueeze(1)
    return centroids

def qni_ccp_feature_perturbation_fixed(model, x, y, centroids, epsilon_q=0.1, target_class=None):
    """
    Direct feature-space perturbation for QNI-CCP.
    Generates adversarial features to be used in the QNI loss term.
    """
    # Temporarily switch to eval mode for gradient calculation stability if needed,
    # but we need gradients flow for S.
    # Note: We keep layers in their current mode but manage gradients manually.
    
    # Step 1: Get original features (Clean)
    # We detach these because we want S relative to the fixed current state
    with torch.no_grad():
        original_features = model.feature_extractor(x)
        original_features = torch.tanh(original_features)
    
    # Create a copy that requires gradients to compute Sensitivity (S)
    features_for_grad = original_features.clone().detach().requires_grad_(True)
    
    # Forward pass through quantum and classifier using the feature copy
    # We use batch processing for speed instead of list comprehension if supported
    try:
        q_out = model.q_layer(features_for_grad)
    except:
        # Fallback to stack if batching issue occurs in TorchLayer
        q_out = torch.stack([model.q_layer(f) for f in features_for_grad])
        
    logits = model.classifier(q_out)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    
    # Check if gradients exist
    if features_for_grad.grad is None:
        S = torch.zeros_like(features_for_grad)
    else:
        S = features_for_grad.grad.data
    
    # Step 2: Select target class and get centroid
    if target_class is None:
        target_classes = []
        for i in range(y.size(0)):
            available_classes = [c for c in range(centroids.size(0)) if c != y[i].item()]
            if available_classes:
                # Pick random class != current
                target_classes.append(random.choice(available_classes))
            else:
                # Fallback if only 1 class exists
                target_classes.append((y[i].item() + 1) % centroids.size(0)) 
        target_class_tensor = torch.tensor(target_classes, device=y.device)
    else:
        target_class_tensor = torch.full_like(y, target_class)
    
    mu_c_prime = centroids[target_class_tensor]
    
    # Step 3: Compute QNI-CCP perturbation
    # x' = x + epsilon_q * [S ⊙ (μ_c' - x)]
    perturbation_direction = mu_c_prime - original_features
    weighted_perturbation = S * perturbation_direction # Element-wise
    
    perturbed_features_final = original_features + epsilon_q * weighted_perturbation
    
    return perturbed_features_final.detach()

# ========== EVALUATION FUNCTION ==========
def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return total_loss / len(dataloader), correct / total

# ========== MAIN EXECUTION ==========
print("Initializing QNI-CCP Training Pipeline...")
model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)

loss_fn = ScheduledFocalLoss(alpha=1, gamma=0.0) # Start with CE behavior

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# Compute Quantum Epsilon once (static circuit)
epsilon_q = compute_quantum_epsilon(n_cnots=N_CNOTS, depth=CIRCUIT_DEPTH, alpha=EPSILON_ALPHA, beta=EPSILON_BETA)
print(f"Computed Quantum Scaling Factor (epsilon_q): {epsilon_q:.6f}")

# Initialize centroids
centroids = compute_class_centroids(model, train_loader, device, num_classes)

train_losses, val_losses = [], []
best_val_loss = float('inf')
epochs_without_improvement = 0
early_stopping_patience = 7

print(f"Starting Training for {num_epochs} epochs with QNI-CCP...")

for epoch in range(num_epochs):
    
    # --- Gamma Scheduling ---
    if epoch < 20:
        loss_mode = "Stage 1: CE (Gamma=0) + QNI"
        loss_fn.gamma = 0.0
    else:
        loss_mode = "Stage 2: Focal (Gamma=1) + QNI"
        loss_fn.gamma = 1.0

    # --- Recompute Centroids Periodically ---
    if epoch > 0 and epoch % 5 == 0:
        centroids = compute_class_centroids(model, train_loader, device, num_classes)
        print(f"  Centroids updated at epoch {epoch}")

    # --- Training Loop ---
    model.train()
    running_loss, running_correct, total_samples = 0.0, 0, 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 1. CLEAN Pass
        optimizer.zero_grad()
        logits_clean = model(inputs)
        loss_clean = loss_fn(logits_clean, labels)
        
        # 2. QNI-CCP Perturbation (Feature Space)
        #    This generates the perturbed features using the logic provided
        perturbed_features = qni_ccp_feature_perturbation_fixed(
            model, inputs, labels, centroids, epsilon_q=epsilon_q
        )
        
        # 3. QNI Pass (Train on perturbed features)
        #    We manually pass features through the rest of the network
        #    Try batch pass first, fallback to stack if needed
        try:
            q_out_p = model.q_layer(perturbed_features)
        except:
            q_out_p = torch.stack([model.q_layer(f) for f in perturbed_features])
            
        logits_qni = model.classifier(q_out_p)
        loss_qni = loss_fn(logits_qni, labels)
        
        # 4. Centroid Regularization
        #    Pull clean features closer to their true class centroid
        with torch.no_grad():
            clean_feats = model.feature_extractor(inputs)
            clean_feats = torch.tanh(clean_feats)
        
        # We need clean_feats attached to graph for loss? 
        # Actually, usually reg is calculated on current forward pass features.
        # Let's re-extract to be safe and graph-connected
        clean_feats_connected = model.feature_extractor(inputs)
        clean_feats_connected = torch.tanh(clean_feats_connected)
        
        current_class_centroids = centroids[labels]
        loss_centroid = F.mse_loss(clean_feats_connected, current_class_centroids)
        
        # 5. Combined Loss
        #    Weights: 0.7 Clean, 0.2 QNI, 0.1 Centroid
        total_loss = (0.7 * loss_clean) + (0.2 * loss_qni) + (0.1 * loss_centroid)
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Stats
        running_loss += total_loss.item()
        running_correct += (logits_clean.argmax(1) == labels).sum().item()
        total_samples += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = running_correct / total_samples
    
    # --- Validation ---
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} | {loss_mode}")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # --- Checkpointing ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "qni_ccp-3.pth")
        print("   💾 Best model saved.")
    else:
        epochs_without_improvement += 1
        print(f"   🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"⏹️ Early stopping triggered after {epoch+1} epochs.")
        break

Using device: cuda
** Dataset loaded successfully **
Class weights calculated.
Initializing QNI-CCP Training Pipeline...
Computed Quantum Scaling Factor (epsilon_q): 0.016393
Computing class centroids...
Starting Training for 70 epochs with QNI-CCP...


Epoch 1/70 | Stage 1: CE (Gamma=0) + QNI
   Train Loss: 2.7532 | Train Acc: 0.0662 | Val Acc: 0.1629
   💾 Best model saved.


Epoch 2/70 | Stage 1: CE (Gamma=0) + QNI
   Train Loss: 2.3340 | Train Acc: 0.1418 | Val Acc: 0.1615
   💾 Best model saved.


Epoch 3/70 | Stage 1: CE (Gamma=0) + QNI
   Train Loss: 2.0969 | Train Acc: 0.1926 | Val Acc: 0.2262
   💾 Best model saved.


Epoch 4/70 | Stage 1: CE (Gamma=0) + QNI
   Train Loss: 1.9703 | Train Acc: 0.2262 | Val Acc: 0.2322
   💾 Best model saved.


...

Epoch 43/70 | Stage 2: Focal (Gamma=1) + QNI
   Train Loss: 0.3931 | Train Acc: 0.8124 | Val Acc: 0.7487
   🕒 No improvement for 3 epoch(s).


Epoch 44/70 | Stage 2: Focal (Gamma=1) + QNI
   Train Loss: 0.3912 | Train Acc: 0.8070 | Val Acc: 0.7143
   🕒 No improvement for 4 epoch(s).


Epoch 45/70 | Stage 2: Focal (Gamma=1) + QNI
   Train Loss: 0.3846 | Train Acc: 0.8142 | Val Acc: 0.7483
   🕒 No improvement for 5 epoch(s).
Computing class centroids...
  Centroids updated at epoch 45


Epoch 46/70 | Stage 2: Focal (Gamma=1) + QNI
   Train Loss: 0.3798 | Train Acc: 0.8181 | Val Acc: 0.7459
   🕒 No improvement for 6 epoch(s).


Epoch 47/70 | Stage 2: Focal (Gamma=1) + QNI
   Train Loss: 0.3503 | Train Acc: 0.8277 | Val Acc: 0.7529
   🕒 No improvement for 7 epoch(s).
⏹️ Early stopping triggered after 47 epochs.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import random
import os

# ========== SEEDING ==========
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)

# ========== DEVICE ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== CONFIGURATION ==========
n_qubits = 10
batch_size = 16
num_classes = 26
num_epochs = 20  # Set to 20 additional epochs
lr = 0.0001      # Lowered slightly for fine-tuning/continuation

# QNI-CCP Specific Config
EPSILON_ALPHA = 1.0
EPSILON_BETA = 1.0
CIRCUIT_DEPTH = 6  
N_CNOTS = 6 * (n_qubits - 1)

# ========== DATA TRANSFORMS ==========
train_transform = transforms.Compose([
    # REVERTED TO GRAYSCALE TO MATCH CHECKPOINT
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ========== DATASET LOADING ==========
TRAIN_PATH = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/train'
VAL_PATH   = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/val'
TEST_PATH  = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH, transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH, transform=eval_transform)
    print("** Dataset loaded successfully **")
    
except Exception as e:
    print(f"Error loading datasets: {e}")
    os.makedirs("dummy_data/train/class1", exist_ok=True)
    os.makedirs("dummy_data/val/class1", exist_ok=True)
    os.makedirs("dummy_data/test/class1", exist_ok=True)
    train_dataset = ImageFolder("dummy_data/train", transform=train_transform)
    val_dataset = ImageFolder("dummy_data/val", transform=eval_transform)
    test_dataset = ImageFolder("dummy_data/test", transform=eval_transform)

# ========== CLASS WEIGHTS ==========
try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(class_weight='balanced',
                                         classes=np.unique(labels),
                                         y=labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print(f"Class weights calculated.")
except:
    print("Warning: Could not calculate class weights. Using ones.")
    class_weights_tensor = torch.ones(num_classes).to(device)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# ========== QUANTUM CIRCUIT ==========
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Angle Embedding
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    
    # Basic Entangler Layers
    for l in range(weights.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (6, n_qubits)}

# ========== MODEL ARCHITECTURE ==========
class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            # REVERTED TO 1 INPUT CHANNEL (GRAYSCALE)
            nn.Conv2d(1, 8, 3, stride=2, padding=1),    
            nn.BatchNorm2d(8), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),   
            nn.BatchNorm2d(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  
            nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), 
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 224, 3, stride=2, padding=1), 
            nn.BatchNorm2d(224), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                
        )
        self.fc_expand = nn.Linear(224, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc_expand(x))
        return self.fc_project(x)

class HybridQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)
        
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.tanh(x)
        q_out = self.q_layer(x) 
        return self.classifier(q_out)

# ========== LOSS FUNCTION ==========
class ScheduledFocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=1.0, reduction='mean'):
        super(ScheduledFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=class_weights_tensor)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ========== QNI-CCP HELPERS ==========
def compute_quantum_epsilon(n_cnots=54, depth=6, alpha=1.0, beta=1.0):
    epsilon_q = 1.0 / (1 + alpha * n_cnots + beta * depth)
    return epsilon_q

def compute_class_centroids(model, loader, device, num_classes):
    model.eval()
    sums = torch.zeros(num_classes, n_qubits, device=device)
    counts = torch.zeros(num_classes, device=device)
    print("Computing class centroids...")
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            features = model.feature_extractor(x)
            features = torch.tanh(features)
            for c in range(num_classes):
                mask = (y == c)
                if mask.any():
                    sums[c] += features[mask].sum(0)
                    counts[c] += mask.sum()
    counts[counts == 0] = 1
    centroids = sums / counts.unsqueeze(1)
    return centroids

def qni_ccp_feature_perturbation_fixed(model, x, y, centroids, epsilon_q=0.1):
    with torch.no_grad():
        original_features = model.feature_extractor(x)
        original_features = torch.tanh(original_features)
    
    features_for_grad = original_features.clone().detach().requires_grad_(True)
    
    try:
        q_out = model.q_layer(features_for_grad)
    except:
        q_out = torch.stack([model.q_layer(f) for f in features_for_grad])
        
    logits = model.classifier(q_out)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    
    if features_for_grad.grad is None:
        S = torch.zeros_like(features_for_grad)
    else:
        S = features_for_grad.grad.data
    
    target_classes = []
    for i in range(y.size(0)):
        available_classes = [c for c in range(centroids.size(0)) if c != y[i].item()]
        if available_classes:
            target_classes.append(random.choice(available_classes))
        else:
            target_classes.append((y[i].item() + 1) % centroids.size(0)) 
    target_class_tensor = torch.tensor(target_classes, device=y.device)
    
    mu_c_prime = centroids[target_class_tensor]
    
    perturbation_direction = mu_c_prime - original_features
    weighted_perturbation = S * perturbation_direction 
    
    perturbed_features_final = original_features + epsilon_q * weighted_perturbation
    return perturbed_features_final.detach()

def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss / len(dataloader), correct / total

# ========== MAIN EXECUTION ==========
print("Initializing QNI-CCP Continuation Training...")
model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)

# 1. LOAD PREVIOUS CHECKPOINT
model_path = "qni_ccp-3_continued.pth"
if os.path.exists(model_path):
    print(f"Loading checkpoint from {model_path}...")
    model.load_state_dict(torch.load(model_path))
else:
    print(f"⚠️ Warning: Checkpoint {model_path} not found. Starting from scratch.")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)

# 2. SET GAMMA=1.0 DIRECTLY (Skipping Stage 1)
loss_fn = ScheduledFocalLoss(alpha=1, gamma=1.0) 

# 3. RE-INIT SCHEDULER
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

epsilon_q = compute_quantum_epsilon(n_cnots=N_CNOTS, depth=CIRCUIT_DEPTH, alpha=EPSILON_ALPHA, beta=EPSILON_BETA)

# 4. RECOMPUTE CENTROIDS FOR LOADED MODEL
centroids = compute_class_centroids(model, train_loader, device, num_classes)

train_losses, val_losses = [], []
best_val_loss = float('inf')

print(f"Continuing training for {num_epochs} more epochs (No Early Stopping)...")

for epoch in range(num_epochs):
    
    # Recompute Centroids Periodically
    if epoch > 0 and epoch % 5 == 0:
        centroids = compute_class_centroids(model, train_loader, device, num_classes)
        print(f"  Centroids updated at epoch {epoch}")

    model.train()
    running_loss, running_correct, total_samples = 0.0, 0, 0
    
    loop = tqdm(train_loader, desc=f"Continued Epoch {epoch+1}/{num_epochs}", leave=False)
    
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 1. CLEAN Pass
        optimizer.zero_grad()
        logits_clean = model(inputs)
        loss_clean = loss_fn(logits_clean, labels)
        
        # 2. QNI-CCP Perturbation
        perturbed_features = qni_ccp_feature_perturbation_fixed(
            model, inputs, labels, centroids, epsilon_q=epsilon_q
        )
        
        # 3. QNI Pass
        try:
            q_out_p = model.q_layer(perturbed_features)
        except:
            q_out_p = torch.stack([model.q_layer(f) for f in perturbed_features])
            
        logits_qni = model.classifier(q_out_p)
        loss_qni = loss_fn(logits_qni, labels)
        
        # 4. Centroid Regularization
        with torch.no_grad():
            clean_feats = model.feature_extractor(inputs)
            clean_feats = torch.tanh(clean_feats)
        
        clean_feats_connected = model.feature_extractor(inputs)
        clean_feats_connected = torch.tanh(clean_feats_connected)
        
        current_class_centroids = centroids[labels]
        loss_centroid = F.mse_loss(clean_feats_connected, current_class_centroids)
        
        # 5. Combined Loss
        total_loss = (0.7 * loss_clean) + (0.2 * loss_qni) + (0.1 * loss_centroid)
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += total_loss.item()
        running_correct += (logits_clean.argmax(1) == labels).sum().item()
        total_samples += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = running_correct / total_samples
    
    # Validation
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1} | Gamma=1.0 + QNI")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Checkpointing (No Early Stopping, just saving best)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "qni_ccp-3_continued.pth")
        print("   💾 New best model saved.")

Using device: cuda
** Dataset loaded successfully **
Class weights calculated.
Initializing QNI-CCP Continuation Training...
Loading checkpoint from qni_ccp-3_continued.pth...
Computing class centroids...
Continuing training for 20 more epochs (No Early Stopping)...


Epoch 1 | Gamma=1.0 + QNI
   Train Loss: 0.3306 | Train Acc: 0.8346 | Val Acc: 0.7604
   💾 New best model saved.


Epoch 2 | Gamma=1.0 + QNI
   Train Loss: 0.3301 | Train Acc: 0.8390 | Val Acc: 0.7669


Epoch 3 | Gamma=1.0 + QNI
   Train Loss: 0.3246 | Train Acc: 0.8372 | Val Acc: 0.7548


Epoch 4 | Gamma=1.0 + QNI
   Train Loss: 0.3278 | Train Acc: 0.8368 | Val Acc: 0.7529


...

Epoch 7 | Gamma=1.0 + QNI
   Train Loss: 0.3422 | Train Acc: 0.8281 | Val Acc: 0.7706


Epoch 8 | Gamma=1.0 + QNI
   Train Loss: 0.3402 | Train Acc: 0.8339 | Val Acc: 0.7669


Epoch 9 | Gamma=1.0 + QNI
   Train Loss: 0.3314 | Train Acc: 0.8340 | Val Acc: 0.7608


Epoch 10 | Gamma=1.0 + QNI
   Train Loss: 0.3290 | Train Acc: 0.8336 | Val Acc: 0.7697
Computing class centroids...
  Centroids updated at epoch 10


Continued Epoch 11/20:  28%|██▊       | 174/622 [00:55<02:18,  3.24it/s]

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import random
import os

# ========== SEEDING ==========
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)

# ========== DEVICE ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== CONFIGURATION ==========
n_qubits = 10
batch_size = 16
num_classes = 26
num_epochs = 20  # Set to 20 additional epochs
lr = 0.0001      # Lowered slightly for fine-tuning/continuation

# QNI-CCP Specific Config
EPSILON_ALPHA = 1.0
EPSILON_BETA = 1.0
CIRCUIT_DEPTH = 6  
N_CNOTS = 6 * (n_qubits - 1)

# ========== DATA TRANSFORMS ==========
train_transform = transforms.Compose([
    # REVERTED TO GRAYSCALE TO MATCH CHECKPOINT
    transforms.Grayscale(1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ========== DATASET LOADING ==========
TRAIN_PATH = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/train'
VAL_PATH   = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/val'
TEST_PATH  = '/home/netsec1/Kathan/MaleVis 2 dataset/malevis_train_val_300x300/test'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH, transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH, transform=eval_transform)
    print("** Dataset loaded successfully **")
    
except Exception as e:
    print(f"Error loading datasets: {e}")
    os.makedirs("dummy_data/train/class1", exist_ok=True)
    os.makedirs("dummy_data/val/class1", exist_ok=True)
    os.makedirs("dummy_data/test/class1", exist_ok=True)
    train_dataset = ImageFolder("dummy_data/train", transform=train_transform)
    val_dataset = ImageFolder("dummy_data/val", transform=eval_transform)
    test_dataset = ImageFolder("dummy_data/test", transform=eval_transform)

# ========== CLASS WEIGHTS ==========
try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(class_weight='balanced',
                                         classes=np.unique(labels),
                                         y=labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print(f"Class weights calculated.")
except:
    print("Warning: Could not calculate class weights. Using ones.")
    class_weights_tensor = torch.ones(num_classes).to(device)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# ========== QUANTUM CIRCUIT ==========
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Angle Embedding
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    
    # Basic Entangler Layers
    for l in range(weights.shape[0]):
        for i in range(n_qubits):
            qml.RY(weights[l][i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (6, n_qubits)}

# ========== MODEL ARCHITECTURE ==========
class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            # REVERTED TO 1 INPUT CHANNEL (GRAYSCALE)
            nn.Conv2d(1, 8, 3, stride=2, padding=1),    
            nn.BatchNorm2d(8), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),   
            nn.BatchNorm2d(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  
            nn.BatchNorm2d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  
            nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), 
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 224, 3, stride=2, padding=1), 
            nn.BatchNorm2d(224), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))                
        )
        self.fc_expand = nn.Linear(224, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc_expand(x))
        return self.fc_project(x)

class HybridQNN(nn.Module):
    def __init__(self, n_qubits, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_qubits)
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)
        
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.tanh(x)
        q_out = self.q_layer(x) 
        return self.classifier(q_out)

# ========== LOSS FUNCTION ==========
class ScheduledFocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=1.0, reduction='mean'):
        super(ScheduledFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=class_weights_tensor)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# ========== QNI-CCP HELPERS ==========
def compute_quantum_epsilon(n_cnots=54, depth=6, alpha=1.0, beta=1.0):
    epsilon_q = 1.0 / (1 + alpha * n_cnots + beta * depth)
    return epsilon_q

def compute_class_centroids(model, loader, device, num_classes):
    model.eval()
    sums = torch.zeros(num_classes, n_qubits, device=device)
    counts = torch.zeros(num_classes, device=device)
    print("Computing class centroids...")
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            features = model.feature_extractor(x)
            features = torch.tanh(features)
            for c in range(num_classes):
                mask = (y == c)
                if mask.any():
                    sums[c] += features[mask].sum(0)
                    counts[c] += mask.sum()
    counts[counts == 0] = 1
    centroids = sums / counts.unsqueeze(1)
    return centroids

def qni_ccp_feature_perturbation_fixed(model, x, y, centroids, epsilon_q=0.1):
    with torch.no_grad():
        original_features = model.feature_extractor(x)
        original_features = torch.tanh(original_features)
    
    features_for_grad = original_features.clone().detach().requires_grad_(True)
    
    try:
        q_out = model.q_layer(features_for_grad)
    except:
        q_out = torch.stack([model.q_layer(f) for f in features_for_grad])
        
    logits = model.classifier(q_out)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    
    if features_for_grad.grad is None:
        S = torch.zeros_like(features_for_grad)
    else:
        S = features_for_grad.grad.data
    
    target_classes = []
    for i in range(y.size(0)):
        available_classes = [c for c in range(centroids.size(0)) if c != y[i].item()]
        if available_classes:
            target_classes.append(random.choice(available_classes))
        else:
            target_classes.append((y[i].item() + 1) % centroids.size(0)) 
    target_class_tensor = torch.tensor(target_classes, device=y.device)
    
    mu_c_prime = centroids[target_class_tensor]
    
    perturbation_direction = mu_c_prime - original_features
    weighted_perturbation = S * perturbation_direction 
    
    perturbed_features_final = original_features + epsilon_q * weighted_perturbation
    return perturbed_features_final.detach()

def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss / len(dataloader), correct / total

# ========== MAIN EXECUTION ==========
print("Initializing QNI-CCP Continuation Training...")
model = HybridQNN(n_qubits=n_qubits, num_classes=num_classes).to(device)

# 1. LOAD PREVIOUS CHECKPOINT
model_path = "qni_ccp-3_continued.pth"
if os.path.exists(model_path):
    print(f"Loading checkpoint from {model_path}...")
    model.load_state_dict(torch.load(model_path))
else:
    print(f"⚠️ Warning: Checkpoint {model_path} not found. Starting from scratch.")

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)

# 2. SET GAMMA=1.0 DIRECTLY (Skipping Stage 1)
loss_fn = ScheduledFocalLoss(alpha=1, gamma=1.0) 

# 3. RE-INIT SCHEDULER
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

epsilon_q = compute_quantum_epsilon(n_cnots=N_CNOTS, depth=CIRCUIT_DEPTH, alpha=EPSILON_ALPHA, beta=EPSILON_BETA)

# 4. RECOMPUTE CENTROIDS FOR LOADED MODEL
centroids = compute_class_centroids(model, train_loader, device, num_classes)

train_losses, val_losses = [], []
best_val_loss = float('inf')

print(f"Continuing training for {num_epochs} more epochs (No Early Stopping)...")

for epoch in range(num_epochs):
    
    # Recompute Centroids Periodically
    if epoch > 0 and epoch % 5 == 0:
        centroids = compute_class_centroids(model, train_loader, device, num_classes)
        print(f"  Centroids updated at epoch {epoch}")

    model.train()
    running_loss, running_correct, total_samples = 0.0, 0, 0
    
    loop = tqdm(train_loader, desc=f"Continued Epoch {epoch+1}/{num_epochs}", leave=False)
    
    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 1. CLEAN Pass
        optimizer.zero_grad()
        logits_clean = model(inputs)
        loss_clean = loss_fn(logits_clean, labels)
        
        # 2. QNI-CCP Perturbation
        perturbed_features = qni_ccp_feature_perturbation_fixed(
            model, inputs, labels, centroids, epsilon_q=epsilon_q
        )
        
        # 3. QNI Pass
        try:
            q_out_p = model.q_layer(perturbed_features)
        except:
            q_out_p = torch.stack([model.q_layer(f) for f in perturbed_features])
            
        logits_qni = model.classifier(q_out_p)
        loss_qni = loss_fn(logits_qni, labels)
        
        # 4. Centroid Regularization
        with torch.no_grad():
            clean_feats = model.feature_extractor(inputs)
            clean_feats = torch.tanh(clean_feats)
        
        clean_feats_connected = model.feature_extractor(inputs)
        clean_feats_connected = torch.tanh(clean_feats_connected)
        
        current_class_centroids = centroids[labels]
        loss_centroid = F.mse_loss(clean_feats_connected, current_class_centroids)
        
        # 5. Combined Loss
        total_loss = (0.7 * loss_clean) + (0.2 * loss_qni) + (0.1 * loss_centroid)
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += total_loss.item()
        running_correct += (logits_clean.argmax(1) == labels).sum().item()
        total_samples += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = running_correct / total_samples
    
    # Validation
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1} | Gamma=1.0 + QNI")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Checkpointing (No Early Stopping, just saving best)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "qni_ccp-3_continued.pth")
        print("   💾 New best model saved.")

Using device: cuda
** Dataset loaded successfully **
Class weights calculated.
Initializing QNI-CCP Continuation Training...
Loading checkpoint from qni_ccp-3_continued.pth...
Computing class centroids...
Continuing training for 20 more epochs (No Early Stopping)...


Epoch 1 | Gamma=1.0 + QNI
   Train Loss: 0.2775 | Train Acc: 0.8633 | Val Acc: 0.7822
   💾 New best model saved.


Epoch 2 | Gamma=1.0 + QNI
   Train Loss: 0.2759 | Train Acc: 0.8669 | Val Acc: 0.7859
   💾 New best model saved.


Epoch 3 | Gamma=1.0 + QNI
   Train Loss: 0.2706 | Train Acc: 0.8673 | Val Acc: 0.7827


Epoch 4 | Gamma=1.0 + QNI
   Train Loss: 0.2798 | Train Acc: 0.8650 | Val Acc: 0.7813


...

Epoch 16 | Gamma=1.0 + QNI
   Train Loss: 0.2615 | Train Acc: 0.8724 | Val Acc: 0.7901


Epoch 17 | Gamma=1.0 + QNI
   Train Loss: 0.2623 | Train Acc: 0.8791 | Val Acc: 0.7711


Epoch 18 | Gamma=1.0 + QNI
   Train Loss: 0.2660 | Train Acc: 0.8766 | Val Acc: 0.7911


Epoch 19 | Gamma=1.0 + QNI
   Train Loss: 0.2612 | Train Acc: 0.8767 | Val Acc: 0.7994


Epoch 20 | Gamma=1.0 + QNI
   Train Loss: 0.2603 | Train Acc: 0.8745 | Val Acc: 0.7939
